# CelebA Smile Detector — Custom CNN + ResNet-18 + Grad-CAM

This notebook trains a binary image classifier to detect smiling faces from the CelebA dataset.
It compares a **custom CNN trained from scratch** against a **fine-tuned ResNet-18**, and uses
**Grad-CAM** to visualize which pixels the model activates when predicting "smiling".

---

## ⚡ Before Running This Notebook

### Step 1 — Enable GPU
Go to `Runtime → Change runtime type → Hardware accelerator → T4 GPU`

### Step 2 — Upload CelebA to Google Drive
1. Download the dataset from [Kaggle — CelebA](https://www.kaggle.com/datasets/jessicali9530/celeba-dataset)
2. You'll get a zip file containing:
   - `img_align_celeba/` (folder of face images)
   - `list_attr_celeba.csv` (attribute labels)
3. Upload both to your Google Drive. Recommended structure:
```
My Drive/
└── CelebA/
    ├── img_align_celeba/
    │   └── img_align_celeba/   ← all .jpg files go here
    └── list_attr_celeba.csv
```

### Step 3 — Update `DRIVE_BASE` path below if needed

### Step 4 — Run All Cells (`Runtime → Run all`)

---

**Expected runtime on T4 GPU:** ~30–40 minutes total
- Data loading: ~5 min
- Custom CNN training (5 epochs): ~5–8 min
- ResNet-18 fine-tuning (5 epochs): ~12–18 min
- Grad-CAM + visualization: ~2 min

## 1. Check GPU & Mount Google Drive

In [ ]:
import torch

# Verify GPU is available
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime → Change runtime type → GPU")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted at /content/drive")

## 2. Configuration

> **Update `DRIVE_BASE`** if you uploaded CelebA to a different folder in your Drive.

In [ ]:
import os

# ── Update this path if your Drive structure is different ─────────────────────
DRIVE_BASE  = '/content/drive/MyDrive/CelebA'
IMAGE_DIR   = os.path.join(DRIVE_BASE, 'img_align_celeba', 'img_align_celeba')
ATTR_CSV    = os.path.join(DRIVE_BASE, 'list_attr_celeba.csv')
# ─────────────────────────────────────────────────────────────────────────────

# Training config
NUM_IMAGES  = 30_000    # number of images to use (out of 202,599 total)
EPOCHS_CNN  = 5         # epochs for custom CNN
EPOCHS_RN18 = 5         # epochs for ResNet-18 (3 frozen + 2 fine-tune)
BATCH_SIZE  = 64
CNN_IMG_SIZE   = 64     # input size for custom CNN
RN18_IMG_SIZE  = 128    # input size for ResNet-18
TARGET_ATTR = 'Smiling'

# Verify paths exist
assert os.path.isdir(IMAGE_DIR), f"Image directory not found: {IMAGE_DIR}"
assert os.path.isfile(ATTR_CSV), f"Attributes CSV not found: {ATTR_CSV}"
print("Paths verified.")
print(f"Image directory: {IMAGE_DIR}")
print(f"Attributes CSV:  {ATTR_CSV}")

## 3. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, models
from torchvision.utils import make_grid
from torch.utils.data import Dataset, DataLoader, random_split

import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print("All imports successful.")

## 4. Dataset

In [ ]:
class CelebADataset(Dataset):
    """
    CelebA dataset wrapper for binary attribute classification.
    Labels stored in list_attr_celeba.csv use +1/-1 encoding;
    we convert to 1/0 for binary cross-entropy.
    """
    def __init__(self, df, image_dir, transform=None, target_attr='Smiling'):
        self.df         = df.reset_index(drop=True)
        self.image_dir  = image_dir
        self.transform  = transform
        self.target_attr = target_attr

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.df.iloc[idx, 0])
        image    = Image.open(img_path).convert('RGB')
        label    = 1 if self.df.iloc[idx][self.target_attr] == 1 else 0

        if self.transform:
            image = self.transform(image)

        return image, label


def get_loaders(df, image_dir, img_size, batch_size, target_attr='Smiling'):
    """Create train/test DataLoaders with appropriate transforms."""
    transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(),       # light augmentation
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3),  # normalize to [-1, 1]
    ])

    dataset    = CelebADataset(df, image_dir, transform, target_attr)
    train_size = int(0.8 * len(dataset))
    test_size  = len(dataset) - train_size
    train_ds, test_ds = random_split(dataset, [train_size, test_size],
                                     generator=torch.Generator().manual_seed(42))

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    return train_loader, test_loader, dataset

print("Dataset classes defined.")

In [ ]:
# Load attribute CSV and take first 30,000 rows
print(f"Loading attributes CSV...")
df_full = pd.read_csv(ATTR_CSV)
df      = df_full.iloc[:NUM_IMAGES].reset_index(drop=True)

smiling_count  = (df[TARGET_ATTR] == 1).sum()
total          = len(df)
print(f"Total images to use : {total:,}")
print(f"Smiling             : {smiling_count:,} ({100*smiling_count/total:.1f}%)")
print(f"Not smiling         : {total - smiling_count:,} ({100*(1-smiling_count/total):.1f}%)")

In [ ]:
# Create data loaders for custom CNN (64×64)
print("Creating CNN data loaders (64×64)...")
cnn_train_loader, cnn_test_loader, cnn_dataset = get_loaders(
    df, IMAGE_DIR, CNN_IMG_SIZE, BATCH_SIZE
)
print(f"Train batches: {len(cnn_train_loader)} ({len(cnn_train_loader.dataset):,} images)")
print(f"Test  batches: {len(cnn_test_loader)} ({len(cnn_test_loader.dataset):,} images)")

In [ ]:
# Visualize a sample of training images
def unnormalize(tensor):
    return (tensor * 0.5 + 0.5).clamp(0, 1)

sample_images, sample_labels = next(iter(cnn_train_loader))
label_names = ['Not Smiling', 'Smiling']

fig, axes = plt.subplots(2, 8, figsize=(16, 5))
for i, ax in enumerate(axes.flatten()):
    img = unnormalize(sample_images[i]).permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.set_title(label_names[sample_labels[i].item()], fontsize=8)
    ax.axis('off')
plt.suptitle('Sample Training Images', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. Custom CNN — Train from Scratch

A lightweight 2-layer CNN trained entirely from random weights on 30K CelebA images.

In [ ]:
class SimpleCNN(nn.Module):
    """
    Two conv layers followed by a small MLP head.
    Layers are named (not in nn.Sequential) so Grad-CAM can hook into conv2.
    """
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.pool1 = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool2 = nn.MaxPool2d(2)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 16 * 16, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.pool1(torch.relu(self.conv1(x)))
        x = self.pool2(torch.relu(self.conv2(x)))
        return self.classifier(x)

cnn_model = SimpleCNN().to(device)
total_params = sum(p.numel() for p in cnn_model.parameters())
print(f"SimpleCNN — {total_params:,} parameters")

In [ ]:
def train_model(model, train_loader, criterion, optimizer, num_epochs, model_name):
    """Train loop — returns list of per-epoch losses."""
    history = []
    model.train()

    for epoch in range(num_epochs):
        running_loss = 0.0
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        avg_loss = running_loss / len(train_loader)
        history.append(avg_loss)
        print(f"[{model_name}] Epoch {epoch+1}/{num_epochs}  Loss: {avg_loss:.4f}")

    return history


def evaluate_model(model, test_loader):
    """Returns accuracy (%) on test set."""
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs    = model(images)
            predicted  = (outputs > 0.5).squeeze().long()
            total     += labels.size(0)
            correct   += (predicted == labels).sum().item()
    return 100 * correct / total

print("Training utilities defined.")

In [ ]:
# Train the custom CNN
cnn_criterion  = nn.BCELoss()
cnn_optimizer  = optim.Adam(cnn_model.parameters(), lr=0.001)

print("Training Custom CNN...\n")
cnn_history = train_model(
    cnn_model, cnn_train_loader, cnn_criterion, cnn_optimizer,
    EPOCHS_CNN, 'Custom CNN'
)

cnn_accuracy = evaluate_model(cnn_model, cnn_test_loader)
print(f"\nCustom CNN Test Accuracy: {cnn_accuracy:.2f}%")

In [ ]:
# Plot training loss curve
plt.figure(figsize=(8, 4))
plt.plot(range(1, EPOCHS_CNN+1), cnn_history, marker='o', color='steelblue', linewidth=2)
plt.title('Custom CNN — Training Loss', fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('BCE Loss')
plt.xticks(range(1, EPOCHS_CNN+1))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Grad-CAM — Visualizing What the CNN Sees

**Grad-CAM** (Gradient-weighted Class Activation Mapping) computes the gradient of the
predicted class score with respect to the last convolutional feature map.
Features that cause the largest gradient signal when activated are the most important
for the prediction — those pixels are highlighted in the heatmap.

For our smile detector, we expect activation around the **mouth and cheek regions**.

In [ ]:
class GradCAM:
    """
    Generates Grad-CAM heatmaps for a given model and target layer.

    Usage:
        cam = GradCAM(model, model.conv2)
        heatmap = cam.generate(image_tensor)  # shape [H, W], values in [0, 1]
    """
    def __init__(self, model, target_layer):
        self.model      = model
        self.gradients  = None
        self.activations = None

        target_layer.register_forward_hook(self._save_activations)
        target_layer.register_full_backward_hook(self._save_gradients)

    def _save_activations(self, module, input, output):
        self.activations = output.detach()

    def _save_gradients(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, x):
        """Returns normalized CAM as a 2D numpy array."""
        self.model.eval()
        self.model.zero_grad()

        output = self.model(x)
        output.backward()   # single sigmoid output — gradient flows directly

        # Global average pool gradients across spatial dimensions
        weights = self.gradients.mean(dim=[2, 3], keepdim=True)
        cam     = torch.relu((weights * self.activations).sum(dim=1)).squeeze()

        cam = cam.cpu().numpy()
        if cam.max() > cam.min():
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        return cam


def overlay_heatmap(image_tensor, cam, alpha=0.45):
    """
    Overlays a Grad-CAM heatmap on the original image.
    image_tensor: normalized tensor [3, H, W]
    cam: 2D numpy array [H, W] with values in [0, 1]
    Returns: numpy image [H, W, 3]
    """
    img = unnormalize(image_tensor).permute(1, 2, 0).numpy()

    # Resize CAM to match image size
    cam_resized = np.array(
        Image.fromarray((cam * 255).astype(np.uint8)).resize(
            (img.shape[1], img.shape[0]), Image.BILINEAR
        )
    ) / 255.0

    # Apply colormap and blend
    heatmap = cm.jet(cam_resized)[:, :, :3]
    overlay = (1 - alpha) * img + alpha * heatmap
    return np.clip(overlay, 0, 1)

print("Grad-CAM class defined.")

In [ ]:
# Attach Grad-CAM to the last conv layer of the custom CNN
cnn_gradcam = GradCAM(cnn_model, cnn_model.conv2)

# Pick 8 test images (mix of smiling / not smiling)
test_images, test_labels = next(iter(cnn_test_loader))

# Select 4 smiling + 4 not smiling
smiling_idx     = [i for i, l in enumerate(test_labels) if l == 1][:4]
not_smiling_idx = [i for i, l in enumerate(test_labels) if l == 0][:4]
selected_idx    = smiling_idx + not_smiling_idx

fig, axes = plt.subplots(2, 8, figsize=(20, 6))
label_names = ['Not Smiling', 'Smiling']

for col, idx in enumerate(selected_idx):
    img_tensor = test_images[idx:idx+1].to(device)
    label      = test_labels[idx].item()

    # Original image
    orig = unnormalize(test_images[idx]).permute(1, 2, 0).numpy()
    axes[0, col].imshow(orig)
    axes[0, col].set_title(label_names[label], fontsize=9,
                           color='green' if label == 1 else 'red')
    axes[0, col].axis('off')

    # Grad-CAM overlay
    cam = cnn_gradcam.generate(img_tensor)
    overlay = overlay_heatmap(test_images[idx], cam)
    axes[1, col].imshow(overlay)
    axes[1, col].set_title('Grad-CAM', fontsize=9)
    axes[1, col].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=10, rotation=90, labelpad=5)
axes[1, 0].set_ylabel('Grad-CAM', fontsize=10, rotation=90, labelpad=5)

plt.suptitle('Custom CNN — Grad-CAM: What activates "Smiling"?',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/CelebA/gradcam_cnn.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved Grad-CAM visualization to Drive.")

## 7. ResNet-18 — Fine-tuning a Pretrained Model

**Transfer learning** uses weights pretrained on ImageNet (1.2M images, 1000 classes)
as a starting point. We replace only the final fully-connected layer and then:

1. **Phase 1 (3 epochs):** Freeze the backbone — only train the new head
2. **Phase 2 (2 epochs):** Unfreeze all layers — fine-tune with a smaller learning rate

This typically converges faster and achieves higher accuracy than training from scratch
because the backbone already understands edges, textures, and facial structure.

In [ ]:
# Create ResNet-18 data loaders (128×128 for better spatial resolution)
print("Creating ResNet-18 data loaders (128×128)...")
rn18_train_loader, rn18_test_loader, _ = get_loaders(
    df, IMAGE_DIR, RN18_IMG_SIZE, BATCH_SIZE
)
print(f"Train batches: {len(rn18_train_loader)} | Test batches: {len(rn18_test_loader)}")

In [ ]:
# Load pretrained ResNet-18 and replace the final layer
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Replace classifier head for binary output
num_features = resnet.fc.in_features
resnet.fc = nn.Sequential(
    nn.Linear(num_features, 1),
    nn.Sigmoid()
)
resnet = resnet.to(device)

total_params     = sum(p.numel() for p in resnet.parameters())
trainable_params = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
print(f"ResNet-18 — {total_params:,} total parameters")
print(f"           {trainable_params:,} trainable at start")

In [ ]:
rn18_criterion = nn.BCELoss()
rn18_history   = []

# ── Phase 1: Train head only (backbone frozen) ─────────────────────────────
print("Phase 1 — Training head only (backbone frozen)...\n")
for param in resnet.parameters():
    param.requires_grad = False
for param in resnet.fc.parameters():
    param.requires_grad = True

phase1_optimizer = optim.Adam(resnet.fc.parameters(), lr=0.001)
history_p1 = train_model(resnet, rn18_train_loader, rn18_criterion,
                         phase1_optimizer, 3, 'ResNet-18 P1')
rn18_history.extend(history_p1)

# ── Phase 2: Fine-tune all layers ──────────────────────────────────────────
print("\nPhase 2 — Fine-tuning all layers (lower learning rate)...\n")
for param in resnet.parameters():
    param.requires_grad = True

phase2_optimizer = optim.Adam(resnet.parameters(), lr=1e-4)
history_p2 = train_model(resnet, rn18_train_loader, rn18_criterion,
                         phase2_optimizer, 2, 'ResNet-18 P2')
rn18_history.extend(history_p2)

rn18_accuracy = evaluate_model(resnet, rn18_test_loader)
print(f"\nResNet-18 Test Accuracy: {rn18_accuracy:.2f}%")

In [ ]:
# Plot ResNet-18 training loss
plt.figure(figsize=(8, 4))
plt.plot(range(1, EPOCHS_RN18+1), rn18_history, marker='o', color='tomato', linewidth=2)
plt.axvline(x=3.5, color='gray', linestyle='--', label='Unfreeze backbone')
plt.title('ResNet-18 — Training Loss', fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('BCE Loss')
plt.xticks(range(1, EPOCHS_RN18+1))
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Grad-CAM on ResNet-18

In [ ]:
# Attach Grad-CAM to the last conv layer of ResNet-18 (layer4[1].conv2)
rn18_gradcam = GradCAM(resnet, resnet.layer4[1].conv2)

# Use same test images but resized for ResNet input
rn18_test_images, rn18_test_labels = next(iter(rn18_test_loader))

smiling_idx     = [i for i, l in enumerate(rn18_test_labels) if l == 1][:4]
not_smiling_idx = [i for i, l in enumerate(rn18_test_labels) if l == 0][:4]
selected_idx    = smiling_idx + not_smiling_idx

fig, axes = plt.subplots(2, 8, figsize=(20, 6))

for col, idx in enumerate(selected_idx):
    img_tensor = rn18_test_images[idx:idx+1].to(device)
    label      = rn18_test_labels[idx].item()

    orig = unnormalize(rn18_test_images[idx]).permute(1, 2, 0).numpy()
    axes[0, col].imshow(orig)
    axes[0, col].set_title(label_names[label], fontsize=9,
                           color='green' if label == 1 else 'red')
    axes[0, col].axis('off')

    cam     = rn18_gradcam.generate(img_tensor)
    overlay = overlay_heatmap(rn18_test_images[idx], cam)
    axes[1, col].imshow(overlay)
    axes[1, col].set_title('Grad-CAM', fontsize=9)
    axes[1, col].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=10, rotation=90, labelpad=5)
axes[1, 0].set_ylabel('Grad-CAM', fontsize=10, rotation=90, labelpad=5)

plt.suptitle('ResNet-18 — Grad-CAM: What activates "Smiling"?',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/CelebA/gradcam_resnet18.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved Grad-CAM visualization to Drive.")

## 9. Results Comparison

In [ ]:
cnn_params  = sum(p.numel() for p in cnn_model.parameters())
rn18_params = sum(p.numel() for p in resnet.parameters())

comparison = pd.DataFrame({
    'Model':        ['Custom CNN', 'ResNet-18 (fine-tuned)'],
    'Test Accuracy': [f'{cnn_accuracy:.2f}%', f'{rn18_accuracy:.2f}%'],
    'Parameters':   [f'{cnn_params:,}', f'{rn18_params:,}'],
    'Input Size':   [f'{CNN_IMG_SIZE}×{CNN_IMG_SIZE}', f'{RN18_IMG_SIZE}×{RN18_IMG_SIZE}'],
    'Training':     ['From scratch', 'ImageNet pretrained'],
})

print("\n" + "="*75)
print("MODEL COMPARISON — CelebA Smile Detector (30K images, 5 epochs)")
print("="*75)
print(comparison.to_string(index=False))
print("="*75)

delta = rn18_accuracy - cnn_accuracy
print(f"\nResNet-18 outperforms Custom CNN by {delta:+.2f} percentage points.")
print("Transfer learning wins because ImageNet pretraining provides rich")
print("visual features (edges, textures, shapes) that transfer directly")
print("to facial attribute recognition.")

In [ ]:
# Bar chart comparison
fig, ax = plt.subplots(figsize=(8, 5))
models_names = ['Custom CNN\n(from scratch)', 'ResNet-18\n(fine-tuned)']
accuracies   = [cnn_accuracy, rn18_accuracy]
colors       = ['steelblue', 'tomato']

bars = ax.bar(models_names, accuracies, color=colors, width=0.4, edgecolor='black', linewidth=0.5)
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{acc:.2f}%', ha='center', fontweight='bold', fontsize=12)

ax.set_ylim(0, 100)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Model Comparison — CelebA Smile Detection\n(30K images, 5 epochs)',
             fontweight='bold', fontsize=13)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/CelebA/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Save Models to Google Drive

In [ ]:
save_dir = os.path.join(DRIVE_BASE, 'saved_models')
os.makedirs(save_dir, exist_ok=True)

# Save Custom CNN
cnn_save_path = os.path.join(save_dir, 'simple_cnn_30k.pth')
torch.save(cnn_model.state_dict(), cnn_save_path)
print(f"Custom CNN saved  → {cnn_save_path}")

# Save ResNet-18
rn18_save_path = os.path.join(save_dir, 'resnet18_finetuned_30k.pth')
torch.save(resnet.state_dict(), rn18_save_path)
print(f"ResNet-18 saved   → {rn18_save_path}")

print("\nAll outputs saved to Google Drive:")
print(f"  {DRIVE_BASE}/saved_models/")
print(f"  {DRIVE_BASE}/gradcam_cnn.png")
print(f"  {DRIVE_BASE}/gradcam_resnet18.png")
print(f"  {DRIVE_BASE}/model_comparison.png")

print(f"\n{'='*50}")
print(f"Final Results:")
print(f"  Custom CNN  accuracy: {cnn_accuracy:.2f}%")
print(f"  ResNet-18   accuracy: {rn18_accuracy:.2f}%")
print(f"{'='*50}")